In [1]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages

In [2]:
class State(TypedDict):
    messages: Annotated[list,add_messages]

graph_builder = StateGraph(State)
graph_builder

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

True

### Simple ChatBot

In [4]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant"
)
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000027BEBC1F380>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000027BEBDB81A0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [5]:
def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

In [6]:
## Adding Node
graph_builder.add_node("llmchatbot",chatbot)

## Adding Edges
graph_builder.add_edge(START,"llmchatbot")
graph_builder.add_edge("llmchatbot",END)

## Compiling the graph
graph = graph_builder.compile()

In [7]:
response = graph.invoke({"messages":"Hi"})
response

{'messages': [HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='6125a308-0b1f-4c66-aee4-375c1cdd0ebb'),
  AIMessage(content="It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 36, 'total_tokens': 59, 'completion_time': 0.033803279, 'completion_tokens_details': None, 'prompt_time': 0.003796564, 'prompt_tokens_details': None, 'queue_time': 0.052978366, 'total_time': 0.037599843}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fc68d-a2d5-7b11-8295-afc2d222d763-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 23, 'total_tokens': 59})]}

In [8]:
response["messages"][-1].content

"It's nice to meet you. Is there something I can help you with or would you like to chat?"

In [9]:
for event in graph.stream({"messages":"Hi, how are you?"}):
    for value in event.values():
        print(value["messages"][-1].content)

I'm functioning properly, thank you for asking. I'm a large language model, so I don't have emotions or feelings like humans do, but I'm here to help you with any questions or tasks you may have. How can I assist you today?


### ChatBot With Tools

In [10]:
from langchain_tavily import TavilySearch
tavily = TavilySearch(max_results=2)
tavily.invoke("What is the capital of France?")

{'query': 'What is the capital of France?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/List_of_capitals_of_France',
   'title': 'List of capitals of France',
   'content': 'This is a chronological list of capitals of France. The capital of France has been Paris since its liberation in 1944.\n\n## Chronology\n\n[edit] [...] Tours (10–13 June 1940), the city served as the temporary capital of France during World War II after the government fled Paris due to the German advance.\n Bordeaux (June 1940), the government was relocated from Paris to Tours then Bordeaux very briefly during World War II, when it became apparent that Paris would soon fall into German hands. [...] Paris (1789–1871), on 5 and 6 October 1789, a throng from Paris invaded the castle and forced the royal family to move back to Paris. The National Constituent Assembly "National Constituent Assembly (France)") followed the King to Paris soon afterward;

In [11]:
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

C:\Users\CSMBD\AppData\Local\Temp\ipykernel_24368\1871147362.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun


In [12]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=2,doc_content_chars_max=500)
arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv,description="Query arxiv papers")

In [13]:
arxiv.invoke("What are the latest papers on quantum computing?")

'Published: 2025-12-16\nTitle: Tianyan: Cloud services with quantum advantage\nAuthors: Tianyan Quantum Group\nSummary: Tianyan Quantum Cloud Platform offers cloud services demonstrating quantum advantage capabilities with a Zuchongzhi 3.0-like superconducting quantum processor. This cloud-accessible superconducting quantum prototype, named Tianyan-287, features 105 qubits and achieves high operational fidelities, with single-qubit gates, two-qubit gates, and readout fidelity at 99.90%, 99.56%, 98.7'

In [21]:
import wikipedia
wikipedia.set_user_agent("MyLangGraphChatbot/1.0 (contact: fahim@asg-security.com)")

In [22]:
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=2,doc_content_chars_max=500)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)

In [23]:
tools = [arxiv,wiki,tavily]

In [24]:
## Initializing the LLM Model
from langchain_groq import ChatGroq
llm = ChatGroq(model="qwen/qwen3.6-27b")

In [25]:
llm.invoke("What is AI?")

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Query**: The user asked "What is AI?" - a broad, foundational question about Artificial Intelligence.\n\n2.  **Identify Key Components of AI**:\n   - Definition: What does AI stand for? What is it fundamentally?\n   - Core concepts: Machine learning, deep learning, natural language processing, computer vision, etc.\n   - Types/Categories: Narrow/Weak AI vs. General/Strong AI (and sometimes Superintelligence)\n   - How it works: Data, algorithms, models, training, inference\n   - Applications: Real-world uses across industries\n   - Current state & limitations: What AI can/cannot do now\n   - Ethical/societal considerations: Bias, transparency, job impact, regulation\n\n3.  **Structure the Response**:\n   - Start with a clear, concise definition\n   - Explain how it works (simplified)\n   - Differentiate types of AI\n   - Give common examples/applications\n   - Mention limitations and ethical consideratio

In [26]:
llm_with_tools = llm.bind_tools(tools=tools)
llm_with_tools.invoke("Find paper 'Attention all you need'")

AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Identify the user\'s request:** The user wants to find the paper titled "Attention all you need" (often referred to as "Attention Is All You Need").\n2.  **Identify the tool:** The `arxiv` tool is perfect for finding academic papers. Alternatively, `tavily_search` or `wikipedia` could work, but `arxiv` is the most direct for papers. I will use `arxiv` with the query "Attention Is All You Need".\n3.  **Formulate the query:** "Attention Is All You Need"\n4.  **Execute tool call:** `arxiv(query="Attention Is All You Need")`\n5.  **Process results:** Extract title, authors, abstract, link, and date from the arxiv response.\n6.  **Draft response:** Present the paper details clearly to the user. Include authors (Vaswani et al.), year (2017), and a brief summary of its significance (introduced the Transformer architecture). Provide the arXiv link.\n\nLet\'s call the tool. \nWait, I should check the exact t

In [27]:
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

def tool_calling_llm(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools=tools))

builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges("tool_calling_llm", tools_condition)
builder.add_edge("tools", END)

graph = builder.compile()

In [28]:
response = graph.invoke({"messages":"Who is Cr7 and find paper 'Attention all you need'?"})

In [ ]:
response

{'messages': [HumanMessage(content="Who is Cr7 ans find paper 'Attention all you need'?", additional_kwargs={}, response_metadata={}, id='4ebd4310-06cc-439d-9f7c-d10977597da1'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n\n1.  **Identify the entities and requests:**\n    *   Entity 1: "Cr7" (Cristiano Ronaldo).\n    *   Entity 2: Paper titled \'Attention all you need\' (Attention Is All You Need).\n    *   Task: Identify who Cr7 is and find the specified paper.\n\n2.  **Analyze "Cr7":**\n    *   Cr7 refers to Cristiano Ronaldo, a famous Portuguese professional footballer.\n    *   I can use the `wikipedia` tool to get a concise summary or just rely on internal knowledge, but using the tool is safer for accuracy and providing a good summary.\n\n3.  **Analyze "Attention all you need":**\n    *   The user is asking for the paper "Attention Is All You Need".\n    *   This is a famous paper in the field of Artificial Intelligence/Machine Learning, in

In [ ]:
response["messages"][-1].content

"Page: Cristiano Ronaldo\nSummary: Cristiano Ronaldo dos Santos Aveiro (born 5 February 1985) is a Portuguese professional footballer who plays as a forward for and captains the Saudi Pro League club Al-Nassr and the Portugal national team. Nicknamed CR7, he is widely regarded as one of the greatest players in history, having won numerous individual accolades throughout his career including five Ballon d'Or awards, a record three UEFA Men's Player of the Year Awards, and four European Golden Shoes"

In [29]:
for val in response["messages"]:
    val.pretty_print()

================================ Human Message =================================

Who is Cr7 and find paper 'Attention all you need'?
================================== Ai Message ==================================
Tool Calls:
  arxiv (qd2z15yys)
 Call ID: qd2z15yys
  Args:
    query: Attention Is All You Need
================================= Tool Message =================================
Name: arxiv

Published: 2021-05-06
Title: Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet
Authors: Luke Melas-Kyriazi
Summary: The strong performance of vision transformers on image classification and other vision tasks is often attributed to the design of their multi-head attention layers. However, the extent to which attention is responsible for this strong performance remains unclear. In this short report, we ask: is the attention layer even necessary? Specifi


### ReACT Agent ChatBot

In [30]:
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

def tool_calling_llm(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools=tools))

builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges("tool_calling_llm", tools_condition)
builder.add_edge("tools", "tool_calling_llm")

graph = builder.compile()

In [31]:
response = graph.invoke({"messages":"Who is Cr7 and find paper 'Attention all you need'?"})

In [32]:
for val in response["messages"]:
    val.pretty_print()

================================ Human Message =================================

Who is Cr7 and find paper 'Attention all you need'?
================================== Ai Message ==================================
Tool Calls:
  wikipedia (79svaw040)
 Call ID: 79svaw040
  Args:
    query: Cristiano Ronaldo
================================= Tool Message =================================
Name: wikipedia

Page: Cristiano Ronaldo
Summary: Cristiano Ronaldo dos Santos Aveiro (born 5 February 1985) is a Portuguese professional footballer who plays as a forward for and captains the Saudi Pro League club Al-Nassr and the Portugal national team. Nicknamed CR7, he is widely regarded as one of the greatest players in history, having won numerous individual accolades throughout his career including five Ballon d'Or awards, a record three UEFA Men's Player of the Year Awards, and four European Golden Shoes
================================== Ai Message ==================================
Tool Calls:


### ChatBot With Memory

In [39]:
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

def tool_calling_llm(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools=tools))

builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges("tool_calling_llm", tools_condition)
builder.add_edge("tools", "tool_calling_llm")

graph = builder.compile(checkpointer=memory)

In [44]:
config = {"configurable": {"thread_id": "1"}}
response = graph.invoke({"messages":"Hi, my name is Fahim Reza"},config=config)
response["messages"][-1].content

"Hello Fahim Reza! It's nice to meet you again. How can I help you today?"

In [45]:
response = graph.invoke({"messages":"Hi, what is my name?"},config=config)
response["messages"][-1].content

'Your name is Fahim Reza.'